# Customer Churn Prediction - Model Tuning

This notebook tunes the baseline models using cross-validation. The goal is to test a small set of useful hyperparameters without making the experiment unnecessarily large.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef
)

data_path = "../data/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(data_path)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

X = df.drop(columns=["customerID", "Churn"])
y = df["Churn"].map({"No": 0, "Yes": 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])


## Logistic Regression search

The regularization strength is varied while the rest of the preprocessing remains unchanged.

In [ ]:
logistic_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

logistic_grid = {
    "model__C": [0.01, 0.1, 1, 10],
    "model__class_weight": [None, "balanced"]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

logistic_search = GridSearchCV(
    logistic_pipeline,
    logistic_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    return_train_score=False
)

logistic_search.fit(X_train, y_train)
print("Best parameters:", logistic_search.best_params_)
print("Best CV F1:", round(logistic_search.best_score_, 4))


## Random Forest search


In [ ]:
forest_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(random_state=42, n_jobs=-1))
])

forest_grid = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_leaf": [1, 2, 5],
    "model__class_weight": [None, "balanced"]
}

forest_search = GridSearchCV(
    forest_pipeline,
    forest_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    return_train_score=False
)

forest_search.fit(X_train, y_train)
print("Best parameters:", forest_search.best_params_)
print("Best CV F1:", round(forest_search.best_score_, 4))


## Evaluate tuned models on the held-out test set

The test set is used only after model selection. This keeps the final evaluation separate from cross-validation.

In [ ]:
tuned_models = {
    "Tuned Logistic Regression": logistic_search.best_estimator_,
    "Tuned Random Forest": forest_search.best_estimator_
}

results = []

for name, model in tuned_models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "MCC": matthews_corrcoef(y_test, y_pred)
    })

tuned_results = pd.DataFrame(results).set_index("Model")
tuned_results.round(4)


## Save the experiment results

This table can be used later when documenting the final model. No model is selected here based on a fixed assumption; the test metrics should be inspected after running the notebook.

In [ ]:
tuned_results.to_csv("../reports/tuned_model_results.csv")
